In [139]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [146]:
gs = pd.read_csv('/Users/k_anisha/Library/CloudStorage/OneDrive-Personal/UCSD/YEAR 2/SPRING 2026/DSC 106/grocerydb.csv')

In [98]:
gs

,name,store,category,brand,FPro,FPro_class,price,price percal,package_weight,Protein,Total Fat,Carbohydrate,"Sugars, total","Fiber, total dietary",Sodium,Cholesterol
0,Stonyfield Organic Whole Milk Strawberry Beet ...,Target,baby-food,Stonyfield,0.815250,3.0,5.29,0.043984,396.893000,5.050505,3.030303,12.121212,9.090909,0.000000,0.080808,0.010101
1,Stonyfield Organic Whole Milk Pear Spinach Man...,Target,baby-food,Stonyfield,0.815250,3.0,5.29,0.043984,396.893000,5.050505,3.030303,12.121212,9.090909,0.000000,0.080808,0.010101
2,Once Upon a Farm Organic Mama Blueberry Fruit ...,Target,baby-food,Once Upon a Farm,0.583219,3.0,2.79,0.055973,90.718400,1.098901,0.549451,13.186813,7.692308,2.197802,0.010989,0.000000
3,Once Upon a Farm Organic Strawberry Kids&#39; ...,Target,baby-food,Once Upon a Farm,0.451056,0.0,2.49,0.019213,90.718400,5.494505,7.692308,15.384615,8.791209,3.296703,0.000000,0.000000
4,Horizon Organic Growing Years Strawberry Kids&...,Target,baby-food,DANNON,0.773519,3.0,4.99,0.017781,396.893000,3.030303,1.010101,14.141414,6.060606,2.020202,0.050505,0.005051
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26245,"Sam's Choice Creamy Honey Almond Butter, 12 oz",Walmart,spread-squeeze,Sam's Choice,0.503963,2.0,4.98,0.002465,340.194000,18.750000,50.000000,25.000000,6.250000,9.375000,0.203125,0.000000
26246,"Nutella and Go Snack Packs, Hazelnut Spread & ...",Walmart,spread-squeeze,Nutella,0.738611,3.0,4.98,NaN,NaN,7.407407,24.074074,66.666667,40.740741,3.703704,0.000000,0.011111
26247,"Sabra Dark Chocolate Dessert Dip & Spread, 8 oz",Walmart,spread-squeeze,Sabra,0.829611,3.0,NaN,NaN,226.796000,3.571429,16.071429,35.714286,21.428571,3.571429,0.142857,0.000000
26248,"MaraNatha, No Stir Peanut Butter, 1.15 oz Packets",Walmart,spread-squeeze,MaraNatha,0.609444,2.0,0.78,0.003828,32.601925,25.000000,53.125000,15.625000,3.125000,6.250000,0.203125,0.000000


In [121]:
# PLOT 1

In [99]:
# Step 1: Let's create the health levels from 0.0 to 1.0, incrementing by 0.05 for each 'level'

health_levels = np.arange(0.0, 1.01, 0.05)

In [100]:
# Step 2: Let's define our constants. 

basket_size = 28 # based on internet searches telling the average size of a weekly family grocery cart
weeks = 52
sim_runs = 5

In [147]:
# Step 3: Manual category mapping. I want the categories to be clearly consolidated to Meat, Dairy, Produce, etc.

def map_category(cat):
    if cat in ['produce-packaged', 'salad']:
        return 'produce'
    elif cat in ['eggs-wf', 'meat-packaged', 'meat-poultry-wf', 
                 'sausage-bacon', 'seafood', 'seafood-wf']:
        return 'meat'
    elif cat in ['cheese', 'dairy-yogurt-drink','milk-milk-substitute']:
        return 'dairy'
    elif cat in ['pasta-noodles', 'produce-beans-wf', 'rice-grains-packaged', 
                 'rice-grains-wf']:
        return 'dry'
    elif cat in ['baking', 'bread', 'muffins-bagels', 'rolls-buns-wraps']:
        return 'baked'
    elif cat in ['prepared-meals-dishes']:
        return 'frozen'
    elif cat in ['nuts-seeds-wf', 'jerky', 'snacks-nuts-seeds']:
        return 'whole_snacks'
    elif cat in ['breakfast', 'cakes', 'cereal', 'cookies-biscuit', 'dressings', 
                 'drink-juice', 'drink-juice-wf', 'drink-shakes-other', 
                 'drink-soft-energy-mixes', 'ice-cream-dessert', 'mac-cheese',
                 'pastry-chocolate-candy', 'pizza', 'pudding-jello', 'sauce-all',
                 'snacks-bars', 'snacks-chips', 'snacks-dips-salsa', 
                 'snacks-mixes-crackers', 'snacks-popcorn']:
        return 'processed_snacks'
    else:
        return 'other'

In [102]:
# Step 4: According to FMI report, this is the 'core' breakdown of grocery items:
"""
produce: 15%
meat: 20%
dairy: 10%
dry: 15%
baked: 5%
flexible: 35%.

We have 28 items per basket, meaining our 'core' basket has a breakdown like so:
produce: 4 items
meat: 6 items
dairy: 3 items
dry: 4 items
baked: 1 items
flexible: 10 items

We can break it down by saying healthy foods are those with FPro scores <= 0.4 and unhealthy are those
with scores >= 0.6. The reason we ignore (0.4, 0.6) is because these are 'average' and don't really have 
a category.

So a 'health score' of 1.0 indicates mostly whole foods, while 0.0 indicates mostly ultra-processed foods.

The health score will be the fraction of items in the basked with FPro < 0.4.

Our healthy pool includes the following categories:
produce, meat, dairy, dry, whole_snacks

Our unhealthy pool includes the following categories:
processed_snacks, frozen

"""

"\nproduce: 15%\nmeat: 20%\ndairy: 10%\ndry: 15%\nbaked: 5%\nflexible: 35%.\n\nWe have 28 items per basket, meaining our 'core' basket has a breakdown like so:\nproduce: 4 items\nmeat: 6 items\ndairy: 3 items\ndry: 4 items\nbaked: 1 items\nflexible: 10 items\n\nWe can break it down by saying healthy foods are those with FPro scores <= 0.4 and unhealthy are those\nwith scores >= 0.6. The reason we ignore (0.4, 0.6) is because these are 'average' and don't really have \na category.\n\nSo a 'health score' of 1.0 indicates mostly whole foods, while 0.0 indicates mostly ultra-processed foods.\n\nThe health score will be the fraction of items in the basked with FPro < 0.4.\n\nOur healthy pool includes the following categories:\nproduce, meat, dairy, dry, whole_snacks\n\nOur unhealthy pool includes the following categories:\nprocessed_snacks, frozen\n\n"

In [148]:
# Step 5: Now let's apply the categories to our dataframe

gs['category_clean'] = gs['category'].apply(map_category)

In [90]:
# Step 6: Let's remove the price outliers, since this is the data we are tracking and there may be some reporting errors in there.

def remove_outliers(group):
    q1 = group['price'].quantile(0.25)
    q3 = group['price'].quantile(0.75)
    iqr = q3 - q1

    return group[(group["price"] >= q1 - 1.5 * iqr) & (group["price"] <= q3 + 1.5 * iqr)]

gs = gs.groupby("store", group_keys = False).apply(remove_outliers)

/var/folders/6c/ldy3_hm15vs21hl7t9yqpl640000gn/T/ipykernel_9619/2012727227.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  gs = gs.groupby("store", group_keys = False).apply(remove_outliers)


In [104]:
# Step 7: This is our sampling function, which we can use to stratify based on store

def sample_from_group(store_df, group, n):
    subset = store_df[store_df['category_clean'] == group]

    if len(subset) == 0:
        return store_df.sample(n = n, replace = True)
    
    return subset.sample(n = n, replace = True)

In [105]:
# Step 8: Now we can build one basket of 28 items

def build_basket(store_df, health_score):
    basket = []
    
    # the fixed items in our basket.
    basket.append(sample_from_group(store_df, "produce", 4))
    basket.append(sample_from_group(store_df, "meat", 6))
    basket.append(sample_from_group(store_df, "dairy", 3))
    basket.append(sample_from_group(store_df, "dry", 4))
    basket.append(sample_from_group(store_df, "baked", 1))
    
    # the flexible items in our basket.
    remaining = 10
    healthy_n = int(health_score * remaining)
    unhealthy_n = remaining - healthy_n
    
    healthy_groups = ["produce", "meat", "dairy", "dry", "whole_snacks"]
    unhealthy_groups = ["processed_snacks", "frozen", "baked"]
    
    for i in range(healthy_n):
        g = np.random.choice(healthy_groups)
        basket.append(sample_from_group(store_df, g, 1))
        
    for i in range(unhealthy_n):
        g = np.random.choice(unhealthy_groups)
        basket.append(sample_from_group(store_df, g, 1))
    
    return pd.concat(basket)

In [106]:
# Step 9: And then simulate the yearly spending, based on price of the items.

def simulate_store(store, df, health_levels):
    store_df = df[df["store"] == store]
    
    results = []
    
    for h in health_levels:
        yearly_costs = []
        
        for i in range(52):
            basket = build_basket(store_df, h)
            weekly_cost = basket["price"].sum()
            yearly_costs.append(weekly_cost * 52)
        
        mean = np.mean(yearly_costs)
        std = np.std(yearly_costs)
        se = std / np.sqrt(len(yearly_costs))
        
        results.append({
            "health_score": h,
            "mean_cost": mean,
            "std": std,
            "se": se
        })
    
    return pd.DataFrame(results)

In [107]:
# Step 10: Finally we can run this simulation.

health_levels = np.arange(0.0, 1.05, 0.05)

walmart = simulate_store("Walmart", gs, health_levels)
target = simulate_store("Target", gs, health_levels)
wf = simulate_store("WholeFoods", gs, health_levels)


In [119]:
import plotly.graph_objects as go
import numpy as np

fig = go.Figure()

def add_store_trace(data, store_name, color):
    x = np.array(data["health_score"])
    y = np.array(data["mean_cost"])
    se = np.array(data["se"])
    
    # Transparent SE area **added first**
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([x, x[::-1]]),
            y=np.concatenate([y + se, (y - se)[::-1]]),
            fill="toself",
            fillcolor=f"rgba({color[0]},{color[1]},{color[2]},0.2)",  # 20% opacity
            line=dict(color="rgba(0,0,0,0)"),  # invisible border
            hoverinfo="skip",
            showlegend=False
        )
    )
    
    # Solid line **on top**
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="lines",
            name=store_name,
            line=dict(color=f"rgb({color[0]},{color[1]},{color[2]})", width=2)
        )
    )

# RGB colors for clarity
add_store_trace(target, "Target", (255,0,0))     # Red
add_store_trace(walmart, "Walmart", (0,0,255))   # Blue
add_store_trace(wf, "Whole Foods", (0,128,0))   # Green

fig.update_layout(
    title=dict(
        text="Cost vs Healthiness of Grocery Basket",
        font=dict(family="Times New Roman", size=20, color="black"),
        x=0.5,
        xanchor="center"
    ),
    xaxis=dict(
        title=dict(
            text="Health Score of Average Family Weekly Grocery Basket",
            font=dict(family="Times New Roman Bold", size=16, color="black")
        ),
        tickfont=dict(family="Times New Roman")
    ),
    yaxis=dict(
        title=dict(
            text="Approximate Yearly Grocery Expenditure ($)",
            font=dict(family="Times New Roman Bold", size=16, color="black")
        ),
        tickprefix="$",
        tickfont=dict(family="Times New Roman")
    ),
    legend=dict(title="Store", font=dict(family="Times New Roman", size=12)),
    font=dict(family="Times New Roman")
)

fig.show()

In [ ]:
# PLOT 2 THIS PLOT SUCKS

In [133]:
# Step 1: Let's identify the optimal macros of a complete meal based on a 2000 calorie/day diet.

"""
calories: 500-600
protein: 25-35g
carbs: 50-70g
fat: 15-25g
sugar: 0-10g
"""

target_protein = 25
target_fat = 20
target_carbs = 50

In [134]:
gs_clean = gs[(gs["Protein"] > 0) & (gs["Total Fat"] > 0) & (gs["Carbohydrate"] > 0)].copy()

In [135]:
# Step 2: Calculate number of calories per package using the nutrition info, then the price per cal
gs_clean['calories_per_100g'] = (4 * gs_clean["Protein"] + 9 * gs_clean["Total Fat"] + 4 * gs_clean["Carbohydrate"])
gs_clean["calories_per_package"] = gs_clean["calories_per_100g"] * (gs_clean["package_weight"] / 100)

In [ ]:
# Step 3: Compute meal cost

# price per gram
gs_clean["price_per_gram"] = gs_clean["price"] / gs_clean["package_weight"]

# macros per gram
gs_clean["protein_per_g"] = gs_clean["Protein"] / 100
gs_clean["fat_per_g"]     = gs_clean["Total Fat"] / 100
gs_clean["carbs_per_g"]   = gs_clean["Carbohydrate"] / 100

# grams needed to hit targets
gs_clean["g_protein"] = target_protein / gs_clean["protein_per_g"]
gs_clean["g_fat"]     = target_fat / gs_clean["fat_per_g"]
gs_clean["g_carbs"]   = target_carbs / gs_clean["carbs_per_g"]

# cost to hit each macro
gs_clean["cost_protein"] = gs_clean["g_protein"] * gs_clean["price_per_gram"]
gs_clean["cost_fat"]     = gs_clean["g_fat"] * gs_clean["price_per_gram"]
gs_clean["cost_carbs"]   = gs_clean["g_carbs"] * gs_clean["price_per_gram"]

# complete meal cost (must satisfy all macros)
gs_clean["meal_cost"] = gs_clean[
    ["cost_protein", "cost_fat", "cost_carbs"]
].max(axis=1)

In [ ]:
# Step 4: Cost Breakdown per Store

cost_breakdown = (
    gs_clean.groupby("store")[["cost_protein", "cost_fat", "cost_carbs"]]
    .median()   # median is still better than mean
    .reset_index()
)

In [144]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

stores = cost_breakdown["store"].unique()

fig = make_subplots(
    rows=1, cols=3,
    specs=[[{"type":"domain"}, {"type":"domain"}, {"type":"domain"}]],
    subplot_titles=[f"<b>{s}</b>" for s in stores]  # bold subplot titles
)

labels = ["Protein", "Fat", "Carbs"]

for i, store in enumerate(stores):
    row = cost_breakdown[cost_breakdown["store"] == store]
    
    values = [
        row["cost_protein"].values[0],
        row["cost_fat"].values[0],
        row["cost_carbs"].values[0]
    ]
    
    fig.add_trace(
        go.Pie(
            labels=labels,
            values=values,
            name=store,
            textinfo="label+percent",
            textfont=dict(size=14),
        ),
        row=1, col=i+1
    )

# Layout styling
fig.update_layout(
    title=dict(
        text="<b>Cost Breakdown of a Balanced Meal by Store</b>",
        x=0.5,
        xanchor="center",
        font=dict(family="Times New Roman", size=22)
    ),
    font=dict(family="Times New Roman", size=14),
    legend=dict(
        title=dict(
            text="<b>Macronutrient</b>",
            font=dict(size=16)
        ),
        font=dict(size=14)
    )
)

# Add total annotations
for i, store in enumerate(stores):
    total = cost_breakdown[
        cost_breakdown["store"] == store
    ][["cost_protein","cost_fat","cost_carbs"]].max(axis=1).values[0]
    
    # Get exact center of each subplot
    domain = fig.get_subplot(1, i+1).x
    
    x_center = (domain[0] + domain[1]) / 2

    fig.add_annotation(
        text=f"<b>Total: ${total:.2f}</b>",
        x=x_center,
        y=-0.1,
        showarrow=False,
        xref="paper",
        yref="paper",
        font=dict(size=14)
    )

fig.show()

In [ ]:
# PLOT 3

In [167]:
import pandas as pd
import plotly.graph_objects as go

# ----------------------------
# Step 1: Count unique brands per category per store
# ----------------------------
variety_df = gs.groupby(['store', 'category_clean'])['brand'].nunique().reset_index()
variety_df.rename(columns={'brand': 'num_brands'}, inplace=True)

# ----------------------------
# Step 2: Pivot for plotting
# ----------------------------
variety_pivot = variety_df.pivot(index='store', columns='category_clean', values='num_brands').fillna(0)

# ----------------------------
# Step 3: Plot stacked bar chart
# ----------------------------
fig = go.Figure()

colors = {
    "produce": "#636EFA",
    "meat": "#EF553B",
    "dairy": "#00CC96",
    "dry": "#AB63FA",
    "baked": "#FFA15A",
    "frozen": "#19D3F3",
    "whole_snacks": "#FF6692",
    "processed_snacks": "#B6E880",
    "other": "#FF97FF"
}

for cat in variety_pivot.columns:
    fig.add_trace(go.Bar(
        x=variety_pivot.index,
        y=variety_pivot[cat],
        name=cat,
        marker_color=colors.get(cat, "#CCCCCC")
    ))

# ----------------------------
# Step 4: Layout styling
# ----------------------------
fig.update_layout(
    barmode='stack',
    title=dict(
        text="<b>Variety of Brands per Category by Store</b>",
        x=0.5,
        xanchor="center",
        font=dict(family="Times New Roman", size=22)
    ),
    xaxis_title="<b>Store</b>",
    yaxis_title="<b>Number of Brands</b>",
    legend_title="<b>Category</b>",
    font=dict(family="Times New Roman", size=14)
)

fig.show()

In [179]:
import pandas as pd
import plotly.graph_objects as go

# Step 1: Count unique brands per category per store
variety_df = gs.groupby(['store', 'category_clean'])['brand'].nunique().reset_index()
variety_df.rename(columns={'brand': 'num_brands'}, inplace=True)

# Step 2: Pivot for plotting
variety_pivot = variety_df.pivot(index='store', columns='category_clean', values='num_brands').fillna(0)

# Step 3: Compute percentages for annotations
percent_pivot = variety_pivot.div(variety_pivot.sum(axis=1), axis=0) * 100

# Step 4: Muted rainbow palette
colors = {
    "produce": "#E27D60",
    "meat": "#F6A67A",
    "dairy": "#FFD28F",
    "dry": "#86C8BC",
    "baked": "#84A9C0",
    "frozen": "#B39BC8",
    "whole_snacks": "#D8A7CA",
    "processed_snacks": "#F2B5D4",
    "other": "#C1D3E0"
}

# Step 5: Plot stacked bars
fig = go.Figure()
for cat in variety_pivot.columns:
    fig.add_trace(go.Bar(
        x=variety_pivot.index,
        y=variety_pivot[cat],
        name=cat,
        marker_color=colors.get(cat, "#CCCCCC")
    ))

# Step 6: Add percentage annotations with arrows
offset = 20
for store in variety_pivot.index:
    cum = 0
    for cat in variety_pivot.columns:
        val = variety_pivot.loc[store, cat]
        if val == 0:
            continue
        y_middle = cum + val / 2
        cum += val
        fig.add_annotation(
            x=store,
            y=y_middle,
            text=f"{percent_pivot.loc[store, cat]:.0f}%",
            showarrow=True,
            arrowhead=2,
            ax=offset,
            ay=0,
            font=dict(family="Times New Roman", size=12, color="black")
        )
        offset *= -1  # alternate left/right

# Step 7: Layout styling
fig.update_layout(
    barmode='stack',
    title=dict(
        text="<b>Variety of Brands per Category by Store</b>",
        x=0.5,
        xanchor="center",
        font=dict(family="Times New Roman", size=22)
    ),
    xaxis_title="<b>Store</b>",
    yaxis_title="<b>Number of Brands</b>",
    legend_title="<b>Category</b>",
    font=dict(family="Times New Roman", size=14)
)

fig.show()